# Manifest Package Operator Tutorial

This notebook teaches the complete workflow for preparing video metadata and later publishing final manifests to ADLS Gen2.

The workflow is intentionally split:

1. **Preparation** discovers and validates videos, calculates hashes, and writes a destination-independent manifest package. It requires no Azure access.
2. **Publication** verifies the package and the exact video bytes, uploads each video, obtains its final URI and ETag, and publishes the final manifest last.

> **Safety:** publication is disabled by default. No cell contacts Azure unless `RUN_PUBLICATION` is explicitly changed to `True`.

## 1. Roles and prerequisites

The same person may perform both roles at different times, or the package may be transferred to another operator.

| Role | Needs source videos | Needs camera catalog | Needs Azure write access |
|---|---:|---:|---:|
| Preparation operator | Yes | Yes | No |
| Publication operator | Yes, unchanged bytes | No | Yes |

Host prerequisites:

- Python 3.10 or newer and `uv`;
- this repository and its installed environment;
- FFmpeg's `ffprobe` on `PATH`;
- for publication only, the `publisher` extra and an Azure identity with narrowly scoped `Storage Blob Data Contributor` access.

Never place credentials in the catalog, exception inventory, package, notebook, command history, or logs.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "samples").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the people-counter repository")


REPO_ROOT = find_repo_root()
VIDEO_ROOT = REPO_ROOT
PARTITION_PREFIX = "samples/"
INPUT_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial-inputs"
PACKAGE_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial"
CATALOG_PATH = INPUT_DIR / "camera_catalog.csv"
EXCEPTION_INVENTORY_PATH = INPUT_DIR / "video-inventory-exceptions.csv"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
assert PACKAGE_DIR.parent == REPO_ROOT / "samples"

print(f"Repository: {REPO_ROOT}")
print(f"Video root: {VIDEO_ROOT}")
print(f"Package output: {PACKAGE_DIR}")

## 2. Create the camera metadata catalog

Create one effective-dated row for each camera configuration, not one row per video. A row defines:

- the source path prefix used to associate videos with a camera;
- stable camera and location identifiers;
- the IANA camera timezone;
- expected frame dimensions;
- the directed counting line;
- how capture time is resolved; and
- the effective UTC interval for that configuration.

Use `capture_time_source=auto` in most cases. Resolution order is:

1. an optional exception-inventory override;
2. embedded stream/container `creation_time`;
3. the configured filename regular expression.

The counting line direction matters: reversing its endpoints reverses `in` and `out`. Coordinates must fit within the declared frame dimensions.

In [ ]:
catalog_fields = [
    "catalog_version", "source_path_prefix", "camera_id", "location_id",
    "camera_timezone", "frame_width", "frame_height",
    "counting_line_x1", "counting_line_y1", "counting_line_x2", "counting_line_y2",
    "capture_time_source", "capture_time_regex", "capture_time_format",
    "effective_from_utc", "effective_to_utc",
]

# The sample clips have different dimensions and effective capture dates.
catalog_rows = [
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3626, "frame_height": 1994,
        "counting_line_x1": 0, "counting_line_y1": 997,
        "counting_line_x2": 3625, "counting_line_y2": 997,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-01-01T00:00:00Z",
        "effective_to_utc": "2023-09-28T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3840, "frame_height": 2160,
        "counting_line_x1": 0, "counting_line_y1": 1080,
        "counting_line_x2": 3839, "counting_line_y2": 1080,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-09-28T00:00:00Z",
        "effective_to_utc": "2023-10-01T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": "samples/",
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 1920, "frame_height": 1080,
        "counting_line_x1": 0, "counting_line_y1": 540,
        "counting_line_x2": 1919, "counting_line_y2": 540,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-10-01T00:00:00Z",
        "effective_to_utc": "",
    },
]

with CATALOG_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=catalog_fields)
    writer.writeheader()
    writer.writerows(catalog_rows)

print(CATALOG_PATH.read_text(encoding="utf-8"))

## 3. Preview source-video metadata

Preparation performs this inspection automatically. The preview below helps an operator understand which timestamps can be derived without manual input.

Do not use filesystem modification time or upload time as recording time. If embedded metadata and filename rules cannot provide a trustworthy timestamp, preparation rejects that video with an actionable reason code.

In [ ]:
video_paths = sorted((REPO_ROOT / "samples").glob("*.mp4"))
if not video_paths:
    raise RuntimeError("No sample MP4 videos were found")

for video_path in video_paths:
    completed = subprocess.run(
        [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries",
            "stream=width,height,duration:stream_tags=creation_time:format=duration:format_tags=creation_time",
            "-of", "json", str(video_path),
        ],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    metadata = json.loads(completed.stdout)
    stream = metadata["streams"][0]
    embedded_time = stream.get("tags", {}).get("creation_time")
    duration = stream.get("duration") or metadata.get("format", {}).get("duration")
    print(
        video_path.relative_to(REPO_ROOT),
        f"{stream['width']}x{stream['height']}",
        f"duration={duration}",
        f"creation_time={embedded_time}",
    )

## 4. Run preparation without exception overrides

The first pass intentionally omits `--inventory`. Two sample videos have embedded timestamps; one does not. Exit code `2` means preparation completed but at least one video was rejected.

An incomplete package is never publishable. Review grouped totals in `summary.json` and individual remediation in `rejection-report.csv`.

In [ ]:
prepare_base = [
    "uv", "run", "prepare-manifests",
    "--catalog", str(CATALOG_PATH),
    "--video-root", str(VIDEO_ROOT),
    "--partition-prefix", PARTITION_PREFIX,
    "--output-dir", str(PACKAGE_DIR),
    "--max-files", "10",
]

first_pass = subprocess.run(
    prepare_base,
    cwd=REPO_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
print(first_pass.stdout.strip())
if first_pass.stderr.strip():
    print(first_pass.stderr.strip())
if first_pass.returncode not in {0, 2}:
    raise RuntimeError(f"Preparation failed with exit code {first_pass.returncode}")
print(f"Exit code: {first_pass.returncode}")

In [ ]:
summary = json.loads((PACKAGE_DIR / "summary.json").read_text(encoding="utf-8"))
with (PACKAGE_DIR / "rejection-report.csv").open(newline="", encoding="utf-8") as handle:
    rejections = list(csv.DictReader(handle))

print(json.dumps(summary, indent=2, sort_keys=True))
print("\nRejections:")
for rejection in rejections:
    print(
        f"- {rejection['video_relative_path']}: {rejection['reason_code']}\n"
        f"  Explanation: {rejection['explanation']}\n"
        f"  Suggested action: {rejection['suggested_action']}"
    )

## 5. Add only exceptional capture-time overrides

Do not manually inventory all videos. The optional inventory exists only for files whose trustworthy capture time cannot be derived by code.

The timestamp below is a tutorial value. In production, obtain it from an authoritative camera or source-system record and include an explicit UTC `Z` or numeric offset.

In [ ]:
exception_rows = [
    {
        "video_relative_path": "samples/three_people_walking.mp4",
        "captured_at_utc": "2023-10-01T01:00:00Z",
    }
]

with EXCEPTION_INVENTORY_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["video_relative_path", "captured_at_utc"],
    )
    writer.writeheader()
    writer.writerows(exception_rows)

print(EXCEPTION_INVENTORY_PATH.read_text(encoding="utf-8"))

In [ ]:
complete_pass = subprocess.run(
    [
        *prepare_base,
        "--inventory", str(EXCEPTION_INVENTORY_PATH),
    ],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(complete_pass.stdout.strip())

## 6. Inspect the completed manifest package

`manifest-package.json` is the authoritative checksum index. Prepared manifests intentionally omit `video_uri` and `source_etag`; publication adds those storage-dependent values later.

Identity is deterministic:

```text
asset_id      = SHA256(normalized source-relative path)
asset_version = SHA256(video bytes)
```

Changing one byte creates a new asset version. Correcting bytes at the same relative path retains the logical asset ID.

In [ ]:
package_index = json.loads(
    (PACKAGE_DIR / "manifest-package.json").read_text(encoding="utf-8")
)
package_summary = json.loads(
    (PACKAGE_DIR / "summary.json").read_text(encoding="utf-8")
)
with (PACKAGE_DIR / "generated-video-inventory.csv").open(
    newline="", encoding="utf-8"
) as handle:
    generated_inventory = list(csv.DictReader(handle))

print("Package files:")
for path in sorted(PACKAGE_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PACKAGE_DIR))

print("\nIndex summary:")
print(json.dumps({
    key: package_index[key]
    for key in ("manifest_package_version", "complete", "discovered", "prepared", "rejected")
}, indent=2))
assert package_index["complete"] is True
assert package_index["prepared"] == len(generated_inventory) == 3
assert not list(PACKAGE_DIR.rglob("*.mp4"))

sample_entry = package_index["entries"][0]
sample_prepared_path = PACKAGE_DIR / sample_entry["prepared_manifest_path"]
sample_prepared = json.loads(sample_prepared_path.read_text(encoding="utf-8"))
assert "video_uri" not in sample_prepared
assert "source_etag" not in sample_prepared
print("\nExample prepared manifest:")
print(json.dumps(sample_prepared, indent=2, sort_keys=True))

In [ ]:
for row in generated_inventory:
    relative_path = row["video_relative_path"]
    video_path = VIDEO_ROOT.joinpath(*relative_path.split("/"))
    expected_asset_id = hashlib.sha256(relative_path.encode("utf-8")).hexdigest()
    video_digest = hashlib.sha256()
    with video_path.open("rb") as source:
        while chunk := source.read(8 * 1024 * 1024):
            video_digest.update(chunk)
    assert row["asset_id"] == expected_asset_id
    assert row["asset_version"] == video_digest.hexdigest()
    assert row["sha256"] == row["asset_version"]
    print(relative_path, row["asset_version"])

print("All generated identities match the source paths and exact video bytes.")

## 7. Validate exactly as publication will

This validation is offline: it reads the package and rehashes every referenced video before any Azure client is created. It catches edited package files, missing videos, changed bytes, unsupported versions, duplicate identities, inconsistent counts, and incomplete preparation.

In [ ]:
from people_counter.manifest_publisher import (
    ManifestPackagePublishConfig,
    validate_manifest_package,
)

validation_config = ManifestPackagePublishConfig(
    manifest_package_dir=PACKAGE_DIR,
    video_root=VIDEO_ROOT,
    storage_account="validation-only",
    filesystem="validation-only",
    staging_prefix="staging",
    incoming_prefix="incoming",
    checkpoint_path=PACKAGE_DIR / "unused-publication.sqlite3",
    rejection_report_path=PACKAGE_DIR / "unused-publication-rejections.csv",
    summary_report_path=PACKAGE_DIR / "unused-publication-summary.json",
    chunk_size=8 * 1024 * 1024,
)

validated_package = validate_manifest_package(validation_config)
print(f"Validated {len(validated_package.plans)} videos without Azure access.")

## 8. Retain or transfer the package

Publication may happen immediately or later. The publication operator needs:

1. the unchanged source-video directory tree; and
2. the complete manifest package directory.

The absolute path may change, but relative paths under `--video-root` must remain the same. SHA-256 integrity is authoritative: publication rehashes the video files and refuses changed or missing bytes.

Do not edit prepared JSON files or `manifest-package.json`. Correct source/catalog data and rerun preparation instead.

## 9. Configure publication access

The publication host needs the optional dependencies:

```bash
uv sync --extra publisher
```

`publish-manifests` uses Azure `DefaultAzureCredential`. Supported credential sources include managed identity, workload identity, an attended Azure CLI login, or environment-provided service-principal variables. Never put secrets in this notebook.

Grant the publisher identity `Storage Blob Data Contributor` on the target filesystem, or equivalent narrowly scoped ADLS permissions for `staging/` and `incoming/`.

Publication validates the entire package and local video hashes before creating the Azure client. It uploads and renames each video first, then creates and renames its final manifest last.

**Before enabling publication:** confirm whether manifest arrival triggers registration in the target deployment. Publishing can start downstream Fabric work; `RUN_PUBLICATION` does not separately gate that automation. Review step 10 before publishing.

In [ ]:
RUN_PUBLICATION = False
STORAGE_ACCOUNT = "replace-with-storage-account"
FILESYSTEM = "replace-with-filesystem"

publish_command = [
    "uv", "run", "--extra", "publisher", "publish-manifests",
    "--manifest-package-dir", str(PACKAGE_DIR),
    "--video-root", str(VIDEO_ROOT),
    "--storage-account", STORAGE_ACCOUNT,
    "--filesystem", FILESYSTEM,
    "--staging-prefix", "staging",
    "--incoming-prefix", "incoming",
    "--checkpoint", str(PACKAGE_DIR / "publication.sqlite3"),
    "--rejection-report", str(PACKAGE_DIR / "publication-rejections.csv"),
    "--summary-report", str(PACKAGE_DIR / "publication-summary.json"),
]

print("Publication command:")
print(" ".join(publish_command))

if RUN_PUBLICATION:
    if STORAGE_ACCOUNT.startswith("replace-") or FILESYSTEM.startswith("replace-"):
        raise ValueError("Set STORAGE_ACCOUNT and FILESYSTEM before enabling publication")
    subprocess.run(publish_command, cwd=REPO_ROOT, check=True)
else:
    print("Publication is disabled. Set RUN_PUBLICATION=True only after reviewing all settings.")

## 10. Verify publication and register backfill work

Publication and SJD registration are separate contracts. `publish-manifests` uploads videos and final source manifests; it does not call Control or create `READY` work. Control does not accept a source-manifest URI or `manifest-package.json` as a registration request.

### 10.1 Verify the published package

1. Require publication exit code `0` and `rejected == 0` in this package's `publication-summary.json`. Require `discovered == planned == published + already_published == len(package_index["entries"])`. On a retry, `already_published` counts successfully verified existing publications.
2. Use the validated plans and `publication.sqlite3` to identify every final video and manifest under `incoming/YYYY/MM/DD/<asset-id>/<asset-version>/`. The manifest filename is the video's stem plus `.json`. Neither the package index nor `prepared-manifests/` is a published source manifest.
3. Read each final manifest and check `asset_id`, `asset_version`, `video_uri`, `source_etag`, `expected_size_bytes`, `expected_sha256`, and `duration_seconds` against the plan and final video. Also check camera/location, capture time, timezone, and counting line. Verify the final video's ETag, size, and SHA-256; object existence alone is insufficient.
4. Keep `incoming/` immutable. Corrected video bytes require a new content-hash asset version; do not overwrite an existing publication.

### 10.2 Select the registration route

- **Arrival-driven deployment:** if the deployed manifest-arrival rule and intake pipeline are configured and verified, observe their registrations first. Match each asset by its Lakehouse source path and source SHA-256. Do not submit a second manual work ID for the same intended processing run. See the [arrival-driven demonstration](customer_end_to_end_demo.ipynb) and its [guide](customer_end_to_end_demo.md).
- **Explicit backfill:** if automatic intake is absent, or for investigated missing registrations, use the deployed **Control SJD** `bulk-register` command below. Coordinate with automatic intake so both routes do not independently register the same work. The seven SJDs alone do not install a manifest-arrival trigger. There is no assumed `pc-backfill-register` pipeline.

Use the target environment's deployed SJD IDs and published Environment, not IDs copied from another environment. Tables must already be initialized by the deployment's Control `bootstrap`. Keep the approved model artifacts available under the attached Lakehouse's `Files/models/` tree. See [production roles and deployment](fabric/README.md) and [canonical SJD binding and release admission](../fabric/spark-job-definitions/README.md).

### 10.3 Translate final manifests into registration records

Expose the published ADLS filesystem through a shortcut in the SJD's default Lakehouse, for example `Files/videos`. Verify that the shortcut targets the filesystem used in step 9 and that the Fabric runtime identity can read it. The publisher's Azure identity permissions do not grant Fabric access automatically.

| Final source manifest / reviewed configuration | Registration field |
|---|---|
| Final video's path below the shortcut | `payload.source_video = /lakehouse/default/Files/videos/incoming/.../<video-name>` |
| `expected_sha256` | `payload.source_sha256` |
| `counting_line` | `payload.line` (four coordinates, preserving direction) |
| `camera_id`, `location_id`, `captured_at_utc`, `camera_timezone` | Same names inside `payload`; preserve source metadata, not submission time |
| `duration_seconds` | Top-level `duration_seconds` (positive, finite) |
| Approved inference profile and artifact hashes | Payload model/runtime settings, top-level `runtime_key` and `config_sha256` |
| Verified installed SDK release manifest digest | Top-level `release_digest` |

**Do not pass the ADLS `video_uri` directly as `source_video`.** The current production localizer requires a `/lakehouse/default/` path and independently verifies `source_sha256`. Keep the source manifest's URI, ETag, size, asset identity, and registration mapping in the backfill evidence. Merely nesting the source manifest as `payload` omits the processing fields and will not produce a valid processing request.

Resolve the release with `resolve_published_release(NotebookUtilsOneLakeFiles().read_text)` in the published Fabric Environment and use its `manifest_sha256`. This verifies the installed SDK's detached manifest and postpublish receipt. The Dispatcher must use the same verified release. Do not substitute the package-index hash, a wheel hash, or a stale copied release digest. Restart stale Spark sessions after an Environment update. Stable registration checks release compatibility for the entire partition before writing work.

Use the [SJD getting-started notebook](fabric/18_sjd_getting_started_demo.ipynb) for the reviewed RT-DETR/OSNet profile, four model artifact paths, release resolution, and SJD submission helpers. Adapt its payload to each final manifest: do not copy its demo work IDs, capture timestamp, or source URIs. Include `line` from the source manifest; `counting_line` is not the worker's field name.

The following is a **template**, not a ready-to-submit partition. Replace all angle-bracket placeholders with verified values. The duration and line illustrate the first tutorial clip; use each video's actual values. Choose persistent work IDs consistent with the deployed intake's identity policy and retain the exact records for retries. A changed payload, configuration, or release cannot reuse an existing immutable work ID.

```json
{
  "items": [
    {
      "work_id": "<persistent-work-id>",
      "payload": {
        "source_video": "/lakehouse/default/Files/videos/incoming/<YYYY/MM/DD>/<asset-id>/<asset-version>/<video-name>",
        "source_sha256": "<expected_sha256>",
        "camera_id": "sample-camera",
        "location_id": "sample-location",
        "captured_at_utc": "<captured_at_utc>",
        "camera_timezone": "UTC",
        "line": [0, 997, 3625, 997],
        "pipeline": "rtdetr-osnet",
        "detector_model": "r18",
        "model_format": "onnx",
        "device_variant": "cpu",
        "batch_size": 1,
        "sample_fps": 1.0,
        "profile_id": "onnx-r18-b1-1fps-1t",
        "runtime_key": "<reviewed-runtime-key>",
        "model_artifact_sha256": {
          "rtdetr_osnet/rtdetr_v2_r18vd/config.json": "<verified-sha256>",
          "rtdetr_osnet/rtdetr_v2_r18vd/preprocessor_config.json": "<verified-sha256>",
          "rtdetr_osnet/rtdetr_v2_r18vd/model.onnx": "<verified-sha256>",
          "rtdetr_osnet/libre_reid_osnet/osnet_ain_x0_25.onnx": "<verified-sha256>"
        }
      },
      "runtime_key": "<reviewed-runtime-key>",
      "duration_seconds": 68.466667,
      "config_sha256": "<reviewed-effective-config-sha256>",
      "release_digest": "<verified-release-manifest-sha256>",
      "max_attempts": 3
    }
  ]
}
```

Validate the completed partition with `people_counter.sjd_control.parse_registration_manifest` before submission. This validates the registration envelope, not source/model availability or inference correctness. The parser accepts an `items` object, a JSON array, or JSONL; prepared package entries are none of these registration records.

### 10.4 Submit bounded partitions to Control

Write each reviewed UTF-8 partition into the target Lakehouse at `Files/people-counter/sjd/v1/intake/backfill/<run-id>/partition-00001.json`. Keep it immutable, retain its exact bytes and SHA-256, and split large backfills into nonempty partitions of at most 10,000 records. Use safe path segments containing letters, digits, dots, underscores, equals signs, or hyphens; start each segment with a letter or digit.

Submit these **command-line arguments to the deployed Control SJD**, not to the local SQLite `pc-control-sjd` command:

```text
bulk-register --partition-path Files/people-counter/sjd/v1/intake/backfill/<run-id>/partition-00001.json --partition-sha256 <64-lowercase-hex-sha256-of-partition-content> --max-items 1000
```

`--max-items` is a rejection bound, not a truncation limit; choose a value from 1 through 10,000 that covers the whole partition. Control verifies the stable intake path, content digest, record count, and immutable registration/release compatibility. `--partition-path` is Lakehouse-relative, not a local package path or ADLS URI. Hash the exact UTF-8 content that Control will read, including whitespace and final newline.

The Fabric submission endpoint is `POST /v1/workspaces/<workspace-id>/sparkJobDefinitions/<control-sjd-id>/jobs/sparkjob/instances`. Put the argument string in `executionData.commandLineArguments`. Poll the returned job instance to `Completed`; a submission response alone is not registration success. For a manually bound workspace, prefix the arguments with `--manual-artifacts <workspace-id> <lakehouse-id> <environment-id>` as described in the [manual deployment guide](../docs/manual-portal-deployment.md); the runtime verifies that binding.

For an isolated single item, Control also accepts `register --work-id <id> --payload-base64 <URL-safe-base64-UTF8-payload-JSON> --runtime-key <key> --duration-seconds <seconds> --config-sha256 <digest> --release-digest <digest> --max-attempts 3`. Encode only the inner `payload` object. Do not supply both `--payload-base64` and `--payload-json`.

Retain the Control job ID and its bulk result: `partition_path`, `partition_sha256`, `item_count`, and `work_ids_sha256`. Verify every intended work ID in `people_counter_sjd_work`. An unchanged retry is idempotent and does not reset work that has already progressed. If registration conflicts, investigate; do not silently generate new IDs or mutate an existing work item's release digest.

### 10.5 Verify processing and reporting completion

Let the deployed Dispatcher claim and invoke Process under the approved schedules. Do not manually `claim` a batch unless you also own the complete process/recovery workflow. Registration alone does not run inference.

Reconcile source videos, package entries, verified publications, and the exact registered work IDs. For each intended successful item require `status == SUCCEEDED` and a non-null `committed_attempt_id` in `people_counter_sjd_work`, then require the matching `(work_id, attempt_id)` in `people_counter_sjd_gold_video`. A `Completed` Spark job is not proof that every video succeeded; inspect attempts and `last_error` for failed or retrying items. Investigate every `DEAD` item and every missing or duplicate registration.

Require successful Reconciliation, Gold, and Refresh runs, no unresolved reconciliation errors, and an empty semantic-refresh outbox before declaring reporting complete. Record any explicitly accepted exclusions separately; do not count dead-lettered work or uncommitted attempt output as a successful backfill.

## 11. Troubleshooting guide

| Reason code | Action |
|---|---|
| `CAPTURE_TIME_MISSING` | Repair embedded metadata, add a filename timestamp rule, or add only affected videos to the exception inventory |
| `CAPTURE_TIME_FILENAME_MISMATCH` | Correct the catalog regex/format or supply an authoritative override |
| `NO_CAMERA_MATCH` | Add or correct the catalog `source_path_prefix` |
| `CAMERA_EFFECTIVE_RANGE_MISMATCH` | Correct the timestamp or effective-dated catalog rows |
| `VIDEO_DIMENSION_MISMATCH` | Correct catalog dimensions/counting line or add another effective-dated configuration |
| `VIDEO_PROBE_FAILED` | Validate, repair, or remux the source video |
| `MANIFEST_PACKAGE_INCOMPLETE` | Resolve all preparation rejections and regenerate the package |
| `MANIFEST_PACKAGE_CHECKSUM_MISMATCH` | Recopy or regenerate the package; never edit prepared files |
| `MANIFEST_PACKAGE_VIDEO_MISMATCH` | Restore the exact source bytes or regenerate from the current video tree |
| `PUBLICATION_CONFLICT` | Investigate the existing ADLS object/checkpoint; never overwrite `incoming/` |

For large backfills, use the grouped rejection counts in `summary.json` to fix the largest issue classes first instead of reviewing console logs one by one.